In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12

In [2]:
# ==========================================
# SETUP (run once in a separate Kaggle cell, with notebook internet ON)
# ==========================================
# !pip install -q segmentation-models-pytorch

import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import segmentation_models_pytorch as smp

# ==========================================
# HYPERPARAMETERS & SETUP
# ==========================================
# NOTE: LEARNING_RATE kept for reference; replaced by ENCODER_LR/DECODER_LR
# below (fix #3 — differential learning rates).
LEARNING_RATE = 1e-4
ENCODER_LR = 1e-5    # fix #3: pretrained encoder moves slowly
DECODER_LR = 1e-4    # fix #3: randomly-init decoder learns faster

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# NOTE: batch_size=8 at 1024x1024 with a ResNet34 decoder can OOM on a
# 16GB Kaggle GPU (T4/P100). If you hit a CUDA out-of-memory error,
# drop this to 4 first before changing anything else.
BATCH_SIZE = 8
NUM_EPOCHS = 10
NUM_WORKERS = 2
IMAGE_HEIGHT = 1024
IMAGE_WIDTH = 1024
PIN_MEMORY = True
LOAD_MODEL = True

ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"

# Kaggle Dataset Paths
TRAIN_IMG_DIR = "/kaggle/input/datasets/amrenderpalarwal/cloudy-data-v2/cloudy_dataset_v2/train/images"
TRAIN_MASK_DIR = "/kaggle/input/datasets/amrenderpalarwal/cloudy-data-v2/cloudy_dataset_v2/train/masks/"
VAL_IMG_DIR = "/kaggle/input/datasets/amrenderpalarwal/cloudy-data-v2/cloudy_dataset_v2/test/images/"
VAL_MASK_DIR = "/kaggle/input/datasets/amrenderpalarwal/cloudy-data-v2/cloudy_dataset_v2/test/masks/"

# ==========================================
# CUSTOM LOSS FUNCTION: BCE + DICE  (kept for reference / easy A-B testing)
# ==========================================
class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=13.0):
        super(DiceBCELoss, self).__init__()
        # Roads are a small minority of pixels (~5-10% in DeepGlobe tiles).
        # pos_weight tells BCE to penalize missed road pixels harder than
        # missed background pixels, so the model can't minimize loss by
        # just predicting "background everywhere."
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight]).to(DEVICE)
        )

    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)

        inputs_sig = torch.sigmoid(inputs)
        inputs_flat = inputs_sig.view(-1)
        targets_flat = targets.view(-1)

        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (
            inputs_flat.sum() + targets_flat.sum() + smooth
        )

        # Dice weighted higher than BCE so the network optimizes for
        # overlapping road shape/structure, not just per-pixel counts.
        return (0.5 * bce_loss) + dice_loss

# ==========================================
# FIX #2: FOCAL TVERSKY LOSS  (now the active loss, see main())
# ==========================================
class FocalTverskyLoss(nn.Module):
    """
    Tversky loss generalizes Dice with separate weights for false positives
    (alpha) and false negatives (beta). For thin, minority-class structures
    like roads, beta > alpha pushes the model to prioritize RECALL — i.e.
    stop missing road pixels — rather than treating FP/FN equally like
    plain Dice does. The gamma exponent (Focal Tversky) further up-weights
    harder examples during training.
    """
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1, pos_weight=13.0):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight]).to(DEVICE)
        )

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)

        probs = torch.sigmoid(inputs).view(-1)
        targets_flat = targets.view(-1)

        TP = (probs * targets_flat).sum()
        FP = ((1 - targets_flat) * probs).sum()
        FN = (targets_flat * (1 - probs)).sum()

        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma

        # Same 0.5 BCE weighting pattern as the original DiceBCELoss — BCE
        # for pixel-level signal, Focal Tversky for structure/recall.
        return (0.5 * bce_loss) + focal_tversky

# ==========================================
# DATASET DEFINITION
# ==========================================
class DeepGlobeDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, self.images[index])
        mask_path = os.path.join(
            self.mask_dir,
            self.images[index].replace(".jpg", ".png").replace("image_", "mask_"),
        )

        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        mask[mask == 255.0] = 1.0

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])

def get_loaders(train_dir, train_maskdir, val_dir, val_maskdir, batch_size,
                 train_transform, val_transform, num_workers=4, pin_memory=True):
    train_ds = DeepGlobeDataset(image_dir=train_dir, mask_dir=train_maskdir, transform=train_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=num_workers,
                               pin_memory=pin_memory, shuffle=True)

    val_ds = DeepGlobeDataset(image_dir=val_dir, mask_dir=val_maskdir, transform=val_transform)
    val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers,
                             pin_memory=pin_memory, shuffle=False)

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    dice_score = 0.0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).unsqueeze(1)

            with torch.amp.autocast('cuda'):
                preds = torch.sigmoid(model(x))
                preds = (preds > 0.5).float()

            num_correct += (preds == y).sum().item()
            num_pixels += torch.numel(preds)
            batch_dice = (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)
            dice_score += batch_dice.item()

    print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}%")
    print(f"Dice score: {dice_score/len(loader):.4f}")
    model.train()

def save_predictions_as_imgs(loader, model, folder="/kaggle/working/", device="cuda"):
    model.eval()
    for idx, (x, y) in enumerate(loader):
        x = x.to(device=device)
        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
        torchvision.utils.save_image(preds, f"{folder}/pred_{idx}.png")
        torchvision.utils.save_image(y.unsqueeze(1), f"{folder}/target_{idx}.png")
    model.train()

# ==========================================
# TRAINING LOGIC
# ==========================================
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().unsqueeze(1).to(device=DEVICE)

        with torch.amp.autocast('cuda'):
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=loss.item())

def main():
    # Get the exact preprocessing stats the pretrained ResNet34 was trained with.
    # IMPORTANT: these come back already in [0,1] scale (e.g. mean=[0.485, ...]).
    # A.Normalize multiplies mean/std by max_pixel_value INTERNALLY, so they
    # must be passed in as-is here — do NOT pre-multiply by 255 yourself,
    # or the normalization gets applied twice and destroys the input signal.
    preprocessing_params = smp.encoders.get_preprocessing_params(
        ENCODER_NAME, pretrained=ENCODER_WEIGHTS
    )
    norm_mean = preprocessing_params["mean"]
    norm_std = preprocessing_params["std"]

    train_transform = A.Compose(
        [
            # FIX #1: mask_interpolation=NEAREST keeps the binary 0/1 mask
            # perfectly crisp through resize/rotate. Default LINEAR blurs
            # thin 1-3px road lines into soft gradients, corrupting the
            # training target itself.
            A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
                      interpolation=cv2.INTER_LINEAR,
                      mask_interpolation=cv2.INTER_NEAREST),
            A.Rotate(limit=35, p=1.0,
                      interpolation=cv2.INTER_LINEAR,
                      mask_interpolation=cv2.INTER_NEAREST),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.Normalize(mean=norm_mean, std=norm_std, max_pixel_value=255.0),
            ToTensorV2(),
        ],
    )

    val_transforms = A.Compose(
        [
            A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
                      interpolation=cv2.INTER_LINEAR,
                      mask_interpolation=cv2.INTER_NEAREST),
            A.Normalize(mean=norm_mean, std=norm_std, max_pixel_value=255.0),
            ToTensorV2(),
        ],
    )

    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        in_channels=3,
        classes=1,
    ).to(DEVICE)

    # FIX #2: Focal Tversky loss instead of plain Dice+BCE — prioritizes
    # recall on thin road pixels via beta > alpha. Swap to DiceBCELoss()
    # here if you want to A-B compare against your previous run.
    loss_fn = FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75, pos_weight=13.0)

    # FIX #3: differential learning rates — pretrained encoder moves slowly,
    # randomly-initialized decoder + segmentation head learn faster. This
    # also tends to reduce loss oscillation caused by one LR being wrong
    # for both parts of the network.
    optimizer = optim.Adam([
        {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
        {'params': model.decoder.parameters(), 'lr': DECODER_LR},
        {'params': model.segmentation_head.parameters(), 'lr': DECODER_LR},
    ])

    train_loader, val_loader = get_loaders(
        TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR,
        BATCH_SIZE, train_transform, val_transforms, NUM_WORKERS, PIN_MEMORY,
    )

    if LOAD_MODEL:
        load_checkpoint(torch.load("/kaggle/input/notebooks/amrenderpalarwal/road-resilience-model-pretrained-encoder-unet-resn/my_checkpoint.pth.tar"), model)

    check_accuracy(val_loader, model, device=DEVICE)

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
        save_checkpoint(checkpoint)

        check_accuracy(val_loader, model, device=DEVICE)
        save_predictions_as_imgs(val_loader, model, folder="/kaggle/working/", device=DEVICE)

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

=> Loading checkpoint
Got 955338056/979369984 with acc 97.55%
Dice score: 0.7453

Epoch [1/10]


100%|██████████| 662/662 [09:22<00:00,  1.18it/s, loss=0.406]


=> Saving checkpoint
Got 953182832/979369984 with acc 97.33%
Dice score: 0.7425

Epoch [2/10]


...

=> Saving checkpoint
Got 953299269/979369984 with acc 97.34%
Dice score: 0.7445

Epoch [9/10]


100%|██████████| 662/662 [09:24<00:00,  1.17it/s, loss=0.433]


=> Saving checkpoint
Got 952469463/979369984 with acc 97.25%
Dice score: 0.7397

Epoch [10/10]


100%|██████████| 662/662 [09:25<00:00,  1.17it/s, loss=0.283]


=> Saving checkpoint
Got 953639524/979369984 with acc 97.37%
Dice score: 0.7473
